# Notebook 01 — Exploratory Data Analysis

**Project:** Concert Ticket Price Predictor  
**Block:** ML Numeric Data  

This notebook explores **Source 1**: the real concert ticket price dataset  
(origin: `github.com/ethanjaredlee/ticketmaster-price-ml`, 1 198 rows).  
Key questions:
- What does the price distribution look like?
- Which features correlate most with ticket price?
- Are there outliers / data quality issues?

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..') / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from data_loader import load_data

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 20)

## 1. Load Data (Source 1 + Source 2 merged)

In [ ]:
df = load_data()
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
print('Data types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())

## 2. Target Variable: minprice (USD)

In [ ]:
print(df['minprice'].describe().round(2))
print(f'\n99th percentile: ${df["minprice"].quantile(0.99):.2f}')
print(f'95th percentile: ${df["minprice"].quantile(0.95):.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Full range
axes[0].hist(df['minprice'], bins=60, color='#4C72B0', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Min. Ticket Price (USD)')
axes[0].set_ylabel('Count')
axes[0].set_title('Price Distribution (full range)')

# Capped at 99th percentile
cap = df['minprice'].quantile(0.99)
axes[1].hist(df[df['minprice'] <= cap]['minprice'], bins=50, color='#55A868', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Min. Ticket Price (USD)')
axes[1].set_title(f'Price Distribution (≤ ${cap:.0f}, 99th pct)')

plt.tight_layout()
plt.savefig('../data/plot_price_dist.png', dpi=120, bbox_inches='tight')
plt.show()

**Finding:** Prices are right-skewed with a long tail (VIP/premium tickets can exceed $500).  
For modeling we keep all rows but the tree models handle skew naturally.

## 3. Categorical Features

In [ ]:
print('Genre counts:')
print(df['genre'].value_counts())
print(f'\nUnique artists: {df["artist"].nunique()}')
print(f'Unique cities:  {df["city"].nunique()}')

In [ ]:
fig = px.box(
    df[df['minprice'] <= df['minprice'].quantile(0.99)],
    x='genre', y='minprice', color='genre',
    title='Min. Ticket Price by Genre',
    labels={'minprice': 'Price (USD)', 'genre': 'Genre'},
)
fig.update_layout(showlegend=False, xaxis_tickangle=-30)
fig.show()

## 4. Numeric Features

In [ ]:
# Artist popularity score distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['score'], bins=20, color='#C44E52', edgecolor='white', alpha=0.85)
axes[0].set_title('Artist Popularity Score')
axes[0].set_xlabel('Score (0–100)')

month_counts = df.groupby('month')['minprice'].median()
axes[1].bar(month_counts.index, month_counts.values, color='#8172B2', alpha=0.85)
axes[1].set_title('Median Price by Month')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Median Price (USD)')

we = df.groupby('weekend')['minprice'].median()
axes[2].bar(['Weekday', 'Weekend'], we.values, color=['#4C72B0', '#DD8452'], alpha=0.85)
axes[2].set_title('Median Price: Weekday vs Weekend')
axes[2].set_ylabel('Median Price (USD)')

plt.tight_layout()
plt.savefig('../data/plot_numeric_features.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Price vs artist score scatter
fig = px.scatter(
    df[df['minprice'] <= df['minprice'].quantile(0.99)],
    x='score', y='minprice', color='genre', opacity=0.5,
    trendline='ols',
    title='Artist Popularity Score vs Ticket Price',
    labels={'score': 'Artist Score', 'minprice': 'Min. Price (USD)'},
)
fig.show()

## 5. Correlation Matrix

In [ ]:
num_cols = ['minprice', 'score', 'pop', 'weekend', 'month']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            linewidths=0.5, annot_kws={'size': 10})
ax.set_title('Feature Correlation Matrix (structured features)')
plt.tight_layout()
plt.savefig('../data/plot_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nCorrelation with minprice:')
print(corr['minprice'].sort_values(ascending=False))

## 6. Key EDA Findings

| Feature | Observation |
|---|---|
| `score` | Strongest positive correlation with price — more popular artists command higher prices |
| `pop` | Larger cities show slightly higher prices (bigger venues, wealthier markets) |
| `weekend` | Weekend events are marginally more expensive |
| `month` | Summer (June–September) dominates the dataset (~83% of events) |
| Genre | Pop, R&B, Dance/Electronic events tend to have higher medians than Metal/Country |
| Price range | $1–$2,999; heavily right-skewed; 95th percentile ≈ $200 |